In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb

file_path = '../data/PGCB_date_power_demand.xlsx'
df = pd.read_excel(file_path)

datetime_col = 'datetime'
df[datetime_col] = pd.to_datetime(df[datetime_col])
df = df.sort_values(datetime_col).reset_index(drop=True)

df['demand_mw'] = df['demand_mw'].ffill()

df['hour'] = df[datetime_col].dt.hour
df['day'] = df[datetime_col].dt.day
df['month'] = df[datetime_col].dt.month

scaler = MinMaxScaler()
df['demand_scaled'] = scaler.fit_transform(df[['demand_mw']])

feature_columns = ['hour', 'day', 'month']
X = df[feature_columns]
y = df['demand_scaled']

split_index = int(len(df) * 0.8)
X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_pred_xgb = xgb_model.predict(X_test)

y_test_inv = scaler.inverse_transform(y_test.values.reshape(-1, 1)).flatten()
y_pred_rf_inv = scaler.inverse_transform(y_pred_rf.reshape(-1, 1)).flatten()
y_pred_xgb_inv = scaler.inverse_transform(y_pred_xgb.reshape(-1, 1)).flatten()

rf_mae = mean_absolute_error(y_test_inv, y_pred_rf_inv)
rf_rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_rf_inv))
xgb_mae = mean_absolute_error(y_test_inv, y_pred_xgb_inv)
xgb_rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_xgb_inv))

print('Random Forest MAE:', round(rf_mae, 2))
print('Random Forest RMSE:', round(rf_rmse, 2))
print('\nXGBoost MAE:', round(xgb_mae, 2))
print('XGBoost RMSE:', round(xgb_rmse, 2))

Random Forest MAE: 3359.87
Random Forest RMSE: 3901.34

XGBoost MAE: 3355.19
XGBoost RMSE: 3895.15
